# 02 — Depression-storage rasters

The 14 per-fabric depstor binary/label rasters under `{data_root}/{FABRIC}/depstor_rasters/`. These are already at fabric extent, so they are shown as decimated thumbnails (no clipping) with a coverage statistic. Missing rasters are skipped with a warning.

> **Run this in JupyterHub on a compute node with enough `--mem`.** Rendering a
> full fabric (especially CONUS `gfv2`, ~361k HRUs) loads large geometries and
> rasters; the login node will run out of memory. Pick the fabric with the
> `FABRIC` env var (or edit the first code cell). Set `SAVE_FIGURES=1` (or
> `viz.SAVE_FIGURES = True`) to write PNGs into `docs/figures/{FABRIC}/`, or
> batch-generate them with `python scripts/render_figures.py --fabric {FABRIC}`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt

from gfv2_params import viz
from gfv2_params.config import load_base_config

# Fabric is selected by the FABRIC env var (set it in your JupyterHub session),
# or edit the default here. SAVE_FIGURES=1 writes PNGs to docs/figures/{FABRIC}/.
FABRIC = os.environ.get("FABRIC", "oregon")
viz.FABRIC = FABRIC
viz.SAVE_FIGURES = os.environ.get("SAVE_FIGURES", "0") == "1"

cfg = load_base_config(fabric=FABRIC)
DATA_ROOT = Path(cfg["data_root"])
HRU_GPKG = Path(cfg["hru_gpkg"])
HRU_LAYER = cfg["hru_layer"]
ID_FEATURE = cfg["id_feature"]

print(f"FABRIC       = {FABRIC}")
print(f"SAVE_FIGURES = {viz.SAVE_FIGURES}  ->  docs/figures/{FABRIC}/")
print(f"data_root    = {DATA_ROOT}")
print(f"hru_gpkg     = {HRU_GPKG} (layer={HRU_LAYER}, id={ID_FEATURE})")

## Inventory

In [ ]:
inventory = viz.depstor_raster_inventory(cfg)
for e in inventory:
    print(f"{e.name:24} {e.path.name}")

## Thumbnails + coverage

`frac>0` is the fraction of valid (non-nodata) cells with a value above zero — e.g. the share of the land mask flagged as a depression, pervious cell, etc. For label rasters (`vpu_id`, `wbody_regions`) it is just the non-zero share.

In [ ]:
import numpy as np

for entry in inventory:
    arr = viz.read_overview(entry.path)
    valid = np.ma.compressed(arr)
    n_valid = int(valid.size)
    frac_set = float((valid > 0).mean()) if n_valid else 0.0
    fig, ax = plt.subplots(figsize=(8, 8))
    viz.plot_raster(ax, arr, cmap=entry.cmap, categorical=True,
                    title=f"{entry.name}  [{FABRIC}]\n"
                          f"valid cells = {n_valid:,}   frac>0 = {frac_set:.3f}",
                    label="class")
    viz.save_figure(fig, f"depstor_{entry.name}")
    plt.show()